In [ ]:
import numpy as np

In [ ]:
X_orig = np.array(([50] * 10 + [5] * 20))
print(X_orig)

In [ ]:
X_orig_mean = np.mean(X_orig)
X_orig_var = np.var(X_orig)
print(X_orig_mean, X_orig_var)

In [ ]:
N = len(X_orig)

K = 10_000  # number of estimates
n = 5  # mini-batch size

In [ ]:
rng = np.random.default_rng(0)
# K mini-batches, each of size n, sampled uniformly with replacement
samples_A = rng.choice(X_orig, size=(K, n), replace=True)
samples_A

In [ ]:
# one estimate per row → K estimates
estimates_A = samples_A.mean(axis=1)
estimates_A

In [ ]:
print("Mean of estimates :", estimates_A.mean())  # ≈ 20
print("Var  of estimates :", estimates_A.var())  # ≈ 450 / n = 90
true_mean = X_orig.mean()  # 20.0
bias_A = estimates_A.mean() - true_mean
print("Bias :", bias_A)

In [ ]:
# Experiment C
# Proposal distribution q: favor 50s
# Each 50 gets 0.07, each 5 gets 0.015 → totals: 10*0.07 + 20*0.015 = 0.7 + 0.3 = 1.0 ✓
q = np.array([0.07] * 10 + [0.015] * 20)
# Sanity check
assert np.isclose(q.sum(), 1.0)
q

In [ ]:
# Sample K mini-batches of size n from q (non-uniform)
idx = rng.choice(N, size=(K, n), replace=True, p=q)
idx

In [ ]:
samples_B = X_orig[idx]

In [ ]:
samples_B

In [ ]:
# Naive estimator: just average (no correction)
estimates_B_naive = samples_B.mean(axis=1)
print("Mean of estimates :", estimates_B_naive.mean())  # expect > 20 (biased high)
print(
    "Var  of estimates :", estimates_B_naive.var()
)  # expect < 90 (lower variance)         # 20.0
bias_B = estimates_B_naive.mean() - true_mean
print("Bias in B :", bias_B)

In [ ]:
# Importance weights: same shape as X_orig, parallel array
w = (1 / N) / q
w

In [ ]:
# Look up the weight for each sampled index
weights_for_samples = w[idx]  # same shape as samples_q
weights_for_samples

In [ ]:
samples_C = weights_for_samples * samples_B
samples_C

In [ ]:
# IS-corrected estimator: average of (weight * value)
estimates_IS = samples_C.mean(axis=1)
estimates_IS

In [ ]:
print("Mean of estimates :", estimates_IS.mean())  # expect ≈ 20
print("Var  of estimates :", estimates_IS.var())  # expect < 90 ← the prize
print("Bias              :", estimates_IS.mean() - true_mean)

In [ ]:
print(1 / (30 * 0.07))
print(1 / (30 * 0.015))
print((0.476 * 50 + 2.222 * 5) / (2 * (0.476 + 2.222)))